In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [2]:
!pip install -q tensorflow-recommenders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 3.0 MB/s eta 0:00:00


In [3]:
import pandas as pd
import pickle
import tensorflow as tf
import tensorflow_recommenders as tfrs

2026-05-14 09:17:53.172996: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778750273.460097      56 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778750273.542127      56 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778750274.187391      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778750274.187446      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778750274.187450      56 computation_placer.cc:177] computation placer alr

In [4]:
training_df = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/train-ds"
)

with open(
    "/kaggle/input/datasets/selinparmar/feature-vocab/feature_vocab.pkl",
    "rb"
) as file:

    feature_vocab = pickle.load(file)

In [5]:
training_df[
    'customer_id'
] = (
    training_df[
        'customer_id'
    ].astype(str)
)

In [10]:
user_vocab = (
    feature_vocab[
        'user_vocab'
    ]
)

favorite_product_vocab = (
    feature_vocab[
        'favorite_product_vocab'
    ]
)

season_vocab = (
    feature_vocab[
        'season_vocab'
    ]
)

In [11]:
class QueryTower(
    tf.keras.Model
):

    def __init__(self):

        super().__init__()

        # Customer ID embedding
        self.customer_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=user_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(user_vocab) + 1,
                32
            )
        ])

        # Favorite product type embedding
        self.favorite_product_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=
                favorite_product_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(
                    favorite_product_vocab
                ) + 1,
                16
            )
        ])

        # Season embedding
        self.season_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=
                season_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(
                    season_vocab
                ) + 1,
                8
            )
        ])

        # Dense network
        self.dense_layers = tf.keras.Sequential([

            tf.keras.layers.Dense(
                128,
                activation='relu'
            ),

            tf.keras.layers.Dense(64)
        ])

    def call(self, features):

        customer_vector = (
            self.customer_embedding(
                features['customer_id']
            )
        )

        favorite_product_vector = (
            self.favorite_product_embedding(
                features[
                    'favorite_product_type'
                ]
            )
        )

        season_vector = (
            self.season_embedding(
                features['season']
            )
        )

        numeric_features = tf.stack([

            tf.cast(
                features['age'],
                tf.float32
            ),

            tf.cast(
                features[
                    'purchase_frequency'
                ],
                tf.float32
            ),

            tf.cast(
                features[
                    'avg_spend'
                ],
                tf.float32
            ),

            tf.cast(
                features[
                    'repeat_purchase_ratio'
                ],
                tf.float32
            )

        ], axis=1)

        return self.dense_layers(
            tf.concat([

                customer_vector,
                favorite_product_vector,
                season_vector,
                numeric_features

            ], axis=1)
        )

In [12]:
query_model = QueryTower()

print(
    "Query Tower built!"
)

Query Tower built!


In [13]:
sample_features = {

    'customer_id':
        tf.constant(
            training_df[
                'customer_id'
            ].head(5)
        ),

    'favorite_product_type':
        tf.constant(
            training_df[
                'favorite_product_type'
            ].head(5)
        ),

    'season':
        tf.constant(
            training_df[
                'season'
            ].head(5)
        ),

    'age':
        tf.constant(
            training_df[
                'age'
            ].head(5),
            dtype=tf.float32
        ),

    'purchase_frequency':
        tf.constant(
            training_df[
                'purchase_frequency'
            ].head(5),
            dtype=tf.float32
        ),

    'avg_spend':
        tf.constant(
            training_df[
                'avg_spend'
            ].head(5),
            dtype=tf.float32
        ),

    'repeat_purchase_ratio':
        tf.constant(
            training_df[
                'repeat_purchase_ratio'
            ].head(5),
            dtype=tf.float32
        )
}

In [14]:
query_embeddings = (
    query_model(
        sample_features
    )
)

print(
    query_embeddings.shape
)

(5, 64)
